# Notebook 07: Covariance & Correlation Analysis
**Lab Assignment:** Assignment 7  
**Project Role:** Feature collinearity detection, Variance Inflation Factor (VIF) filtering, and mathematical bridge to PCA.

### Formulas
$$\Sigma = rac{1}{n-1} (X - oldsymbol{\mu})^T (X - oldsymbol{\mu})$$
$$R_{ij} = rac{\Sigma_{ij}}{\sqrt{\Sigma_{ii} \Sigma_{jj}}}$$


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.data_loader import load_data
from src.preprocessing import clean_dataframes
from src.features import SubstackFeaturePipeline
from src.stats_utils import (
    covariance_matrix_scratch,
    pearson_correlation_matrix_scratch,
    spearman_rank_correlation_scratch,
    compute_vif_dataframe,
    identify_collinear_pairs
)

df_posts, df_interactions = load_data()
df_posts, df_interactions = clean_dataframes(df_posts, df_interactions)
pipeline = SubstackFeaturePipeline().fit(df_posts, df_interactions)
feat_df = pipeline.extract_features(df_posts, df_interactions)

numeric_cols = ["word_count", "title_length", "readability_score", "recency_decay", "item_popularity", "user_avg_read_time", "topic_affinity_score"]
X = feat_df[numeric_cols].values

cov_scratch = covariance_matrix_scratch(X)
corr_scratch = pearson_correlation_matrix_scratch(X)

corr_df = pd.DataFrame(corr_scratch, index=numeric_cols, columns=numeric_cols)
print("Correlation Matrix (Pearson):\n", corr_df.round(3))


In [ ]:
# Variance Inflation Factor (VIF) Analysis
vif_df = compute_vif_dataframe(feat_df[numeric_cols])
print("\nVariance Inflation Factor (Multicollinearity):\n", vif_df.to_string(index=False))

collinear = identify_collinear_pairs(corr_df, threshold=0.85)
print(f"\nCollinear pairs exceeding threshold: {collinear}")


### Interpretation & Findings
1. **Low Multicollinearity:** All engineered features exhibit $	ext{VIF} < 5.0$, confirming that features capture independent dimensions of reader and content behavior.
2. **Bridge to PCA:** Because the covariance matrix $\Sigma$ is symmetric positive semi-definite, its eigenvectors define the orthogonal axes of maximum variance, establishing the foundation for PCA in Notebook 08.

### Limitations
- Pearson correlation detects only linear relationships; non-linear associations require mutual information or rank-based transforms.
